# Folding the masked nanobody construct

Companion notebook to **[dtu_postdoc](https://github.com/Natalija-Stepurko/dtu_postdoc)**.

The CPU stages of the pipeline settle the *geometry* and the *thermodynamics* of a
tethered mask on VHH TPL1158_01_C09 (PDB 9GCN). They cannot say whether a given
mask sequence actually folds back and covers the paratope. That is a structure-prediction
question and it needs a GPU.

The construct `mask – linker – VHH` is a **single polypeptide chain**, which is exactly
what ESMFold predicts from sequence alone, with no MSA. At ~160 residues it runs in
seconds on a free Colab T4, where AlphaFold 3, Boltz-2, RFdiffusion and BindCraft
would not fit comfortably.

**What this notebook tests:** does the mask occlude the paratope, and does occlusion
depend on linker length and on mask identity in the way the thermodynamic model predicts?

**Controls are built in.** `epi1` grafts α-cobratoxin's own VHH-contacting segment
(loop II tip, RGKRV). `epi1_scram` is the same amino acids shuffled, `epi1_half` is
truncated, and `polyG` has no designed interaction at all. If occlusion is real,
`epi1` should separate from `polyG`; if ESMFold is just reporting that a tethered
peptide lies near its anchor, they will not separate.

**Runtime:** ~3 min setup, ~5-10 min folding. Set the runtime to **T4 GPU**.

## 1. Check the GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "no GPU found")
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"

## 2. Install

In [ ]:
%pip -q install "transformers==4.46.3" accelerate biotite py3Dmol 2>&1 | tail -1

## 3. Load the construct library

Built by `src/s05_constructs.py` in the repo, straight from the deposited structure.

In [ ]:
import json, urllib.request, numpy as np, pandas as pd

RAW = "https://raw.githubusercontent.com/Natalija-Stepurko/dtu_postdoc/main"
lib  = json.loads(urllib.request.urlopen(f"{RAW}/results/s05_constructs.json").read())
iface = json.loads(urllib.request.urlopen(f"{RAW}/results/s02_interface.json").read())

constructs = lib["constructs"]
PARATOPE   = set(lib["paratope_labels"])
VHH        = lib["vhh_sequence"]
print(f"{len(constructs)} constructs, {len(VHH)}-residue VHH, "
      f"{len(PARATOPE)} paratope residues")
pd.DataFrame([{k: c[k] for k in ("id","mask_kind","mask_seq","linker_residues","length")}
              for c in constructs]).head(30)

## 4. Load ESMFold

In [ ]:
import torch
from transformers import AutoTokenizer, EsmForProteinFolding

tok = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
model = model.cuda().eval()
model.esm = model.esm.half()            # fp16 language-model trunk fits a T4
torch.backends.cuda.matmul.allow_tf32 = True
model.trunk.set_chunk_size(64)          # lower memory, slightly slower
print("ESMFold ready")

## 5. Fold every construct

Single sequence in, structure out. The free VHH is folded too and serves as the
reference for both the paratope-exposure baseline and the fold-preservation check.

In [ ]:
import time
from transformers.models.esm.openfold_utils.protein import to_pdb, Protein as OFProtein
from transformers.models.esm.openfold_utils.feats import atom14_to_atom37

def fold(seq):
    with torch.no_grad():
        inp = tok([seq], return_tensors="pt", add_special_tokens=False)["input_ids"].cuda()
        out = model(inp)
    final = atom14_to_atom37(out["positions"][-1], out)
    out = {k: v.cpu().numpy() for k, v in out.items()}
    final = final.cpu().numpy()
    p = OFProtein(aatype=out["aatype"][0], atom_positions=final[0],
                  atom_mask=out["atom37_atom_exists"][0],
                  residue_index=out["residue_index"][0] + 1,
                  b_factors=out["plddt"][0],
                  chain_index=out["chain_index"][0] if "chain_index" in out else None)
    return to_pdb(p), out["plddt"][0, :, 1]     # CA pLDDT per residue

pdbs, plddts = {}, {}
t0 = time.time()
for i, c in enumerate(constructs, 1):
    pdbs[c["id"]], plddts[c["id"]] = fold(c["sequence"])
    print(f"[{i:2d}/{len(constructs)}] {c['id']:16s} {c['length']:3d} aa  "
          f"pLDDT {plddts[c['id']].mean():5.1f}  ({time.time()-t0:5.1f}s)")
print(f"\ndone in {time.time()-t0:.0f}s")

## 6. Measure paratope occlusion

For each construct, compute the solvent-accessible surface of the paratope residues
inside the fusion and compare with the same residues in the folded free VHH. A mask
that works buries them; a mask that does nothing leaves them exposed.

In [ ]:
import io, biotite.structure as struc, biotite.structure.io.pdb as biopdb

def load(pdb_str):
    return biopdb.PDBFile.read(io.StringIO(pdb_str)).get_structure(model=1)

def res_sasa(a):
    v = np.nan_to_num(struc.sasa(a, probe_radius=1.4, point_number=300))
    st = struc.get_residue_starts(a)
    out = np.zeros(len(st))
    idx = np.zeros(a.array_length(), dtype=int)
    for n, s in enumerate(st):
        e = st[n+1] if n+1 < len(st) else a.array_length()
        idx[s:e] = n
    np.add.at(out, idx, v)
    return out

# paratope positions as indices into the VHH, from the deposited numbering
vhh_labels = iface["chain_pairs"]["A"]["vhh_residue_labels"]
para_idx = np.array([i for i, l in enumerate(vhh_labels) if l in PARATOPE])

free = load(pdbs["VHH_only"])
free = free[struc.filter_amino_acids(free)]
free_sasa = res_sasa(free)[para_idx]

rows = []
for c in constructs:
    if c["id"] == "VHH_only":
        continue
    a = load(pdbs[c["id"]]); a = a[struc.filter_amino_acids(a)]
    off = c["vhh_range"][0] - 1                    # VHH starts after mask+linker
    s = res_sasa(a)[off + para_idx]
    occl = 1 - s.sum() / free_sasa.sum()
    # mask-to-paratope closest approach
    ca = a[a.atom_name == "CA"]
    mi = np.arange(c["mask_range"][0]-1, c["mask_range"][1])
    pi = off + para_idx
    d = np.linalg.norm(ca.coord[mi][:, None] - ca.coord[pi][None], axis=-1)
    pl = plddts[c["id"]]
    rows.append({"id": c["id"], "mask": c["mask"], "kind": c["mask_kind"],
                 "linker": c["linker_residues"],
                 "occlusion": round(float(occl), 4),
                 "paratope_sasa_A2": round(float(s.sum()), 1),
                 "min_mask_paratope_CA_A": round(float(d.min()), 2),
                 "plddt_all": round(float(pl.mean()), 2),
                 "plddt_vhh": round(float(pl[off:].mean()), 2),
                 "plddt_mask": round(float(pl[mi].mean()), 2)})

df = pd.DataFrame(rows).sort_values(["mask", "linker"]).reset_index(drop=True)
print(f"free-VHH paratope SASA: {free_sasa.sum():.1f} A2")
df

## 7. Results

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi":140,"font.size":9,"axes.spines.top":False,
                     "axes.spines.right":False,"axes.grid":True,"grid.alpha":.25,
                     "legend.frameon":False,"savefig.bbox":"tight"})
COL = {"epi1":"#2a78d6","epi1_scram":"#eb6834","epi1_half":"#1baf7a",
       "epi2":"#eda100","polyG":"#52514e"}

fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.5))
ax = axes[0]
for m, g in df.groupby("mask"):
    g = g.sort_values("linker")
    ax.plot(g.linker, g.occlusion*100, marker="o", ms=5, lw=2, color=COL[m])
    ax.annotate(m, (g.linker.iloc[-1], g.occlusion.iloc[-1]*100),
                textcoords="offset points", xytext=(5,0), fontsize=8, color=COL[m])
ax.set_xlabel("linker length (residues)"); ax.set_ylabel("paratope occluded (%)")
ax.set_title("Does the mask cover the paratope?"); ax.set_xlim(2, 38)

ax = axes[1]
for m, g in df.groupby("mask"):
    ax.scatter(g.plddt_vhh, g.occlusion*100, s=38, color=COL[m], label=m,
               edgecolor="white", linewidth=1.2)
ax.set_xlabel("VHH-region pLDDT"); ax.set_ylabel("paratope occluded (%)")
ax.set_title("Occlusion against fold confidence"); ax.legend(fontsize=7.5, loc="best")
plt.show()

print(df.groupby("kind")[["occlusion","min_mask_paratope_CA_A","plddt_vhh"]]
        .agg(["mean","std"]).round(3))

### Read this honestly

ESMFold returns one structure per sequence. A tethered peptide lying against the
paratope in that single prediction is **a hypothesis about the masked state**, not a
measurement of masking efficiency and not a $K_D$. pLDDT is a confidence measure, not
a folding free energy or a probability of expressing.

The comparison that carries information is the **contrast between the grafted mask and
its scrambled and poly-glycine controls** at matched linker length. If the graft does
not separate from `polyG`, the prediction is reporting tether geometry rather than a
designed interaction.

## 8. Look at one structure

In [ ]:
import py3Dmol
pick = "epi1_L5"
c = next(x for x in constructs if x["id"] == pick)
v = py3Dmol.view(width=720, height=480)
v.addModel(pdbs[pick], "pdb")
v.setStyle({"cartoon": {"color": "#d8d7d2"}})
v.setStyle({"resi": f"{c['vhh_range'][0]}-{c['vhh_range'][1]}"}, {"cartoon": {"color": "#2a78d6"}})
v.setStyle({"resi": f"{c['mask_range'][0]}-{c['mask_range'][1]}"},
           {"cartoon": {"color": "#1baf7a"}, "stick": {"color": "#1baf7a"}})
off = c["vhh_range"][0] - 1
v.addStyle({"resi": [int(i)+off+1 for i in para_idx]},
           {"stick": {"color": "#eb6834"}})
v.zoomTo(); v.show()
print(f"{pick}: VHH blue, mask green, paratope orange sticks")

## 9. Save results back

In [ ]:
import json
from google.colab import files
out = {"constructs": rows,
       "free_vhh_paratope_sasa_A2": float(free_sasa.sum()),
       "model": "facebook/esmfold_v1", "note": "single-sequence prediction, no MSA"}
with open("esmfold_results.json", "w") as f:
    json.dump(out, f, indent=2)
for cid, p in pdbs.items():
    open(f"{cid}.pdb", "w").write(p)
!zip -qr esmfold_structures.zip *.pdb
files.download("esmfold_results.json")
print("also available: esmfold_structures.zip")